In [1]:
import matplotlib
import platform
print ("Operating system: ", platform.system())
if "Linux" in platform.system():
    %matplotlib tk
else:
    %matplotlib qt
    

import matplotlib.pyplot as plt
%autosave 180
%load_ext autoreload
%autoreload 2
import numpy as np

from scipy import ndimage as ndi
from skimage.segmentation import watershed
from skimage.feature import peak_local_max
import scipy
import os
import time
from tqdm import trange, tqdm



Operating system:  Linux


Autosaving every 180 seconds


In [2]:
##########################################################
##########################################################
##########################################################
#
def phase_correlation(a, b):
    G_a = np.fft.fft2(a)
    G_b = np.fft.fft2(b)
    conj_b = np.ma.conjugate(G_b)
    R = G_a*conj_b
    R /= np.absolute(R)
    r = np.fft.ifft2(R).real
    return r

#
def pad_data(img):
    img[:50] = 0
    img[-50:] = 0
    img[:,:50] = 0
    img[:,-50:] = 0

    return img

#
def get_drift_xy(data, 
                 x_drift_max, 
                 y_drift_max):

    # make random shift array
    x_shifts = np.random.randint(low=-x_drift_max,
                                 high = x_drift_max,
                                 size = data.shape[0])
    y_shifts = np.random.randint(low=-x_drift_max,
                                 high = x_drift_max,
                                 size = data.shape[0])

    return x_shifts, y_shifts


def find_best_correlation_map(data,
                              n_imgs_to_sample = 500,
                             n_best_imgs = 100):
    # find best correlation map first
    #n_imgs_template = 500
    idx_imgs = np.random.choice(np.arange(data.shape[0]),
                                n_imgs_to_sample,
                                replace=False)

    # make temporary template to match to
    template = np.mean(data[idx_imgs],axis=0)

    #
    corr_maxs = np.zeros(idx_imgs.shape[0])
    ctr=0
    for k in tqdm(idx_imgs, desc="computing phase correlations"):

        #
        temp = phase_correlation(data[k], template)

        r,c = np.unravel_index(temp.argmax(), temp.shape)

        maxcorr = temp[r,c]
        corr_maxs[ctr] = maxcorr
        ctr+=1
    #
    idx = np.argsort(corr_maxs)[::-1]

    # take the n best images and compute template
    idx_best = idx[:n_best_imgs]
    template = data[idx_imgs[idx_best]].mean(0)

    #
    return corr_maxs, template, idx_imgs




def phase_correlation_parallel2(idx_parmap,
                               template,
                               fname):
    # load the data as mmap;
    # - this should avoid memmory crash issues
    # TODO: can even have this mmap reset every 1000 frames so that
    #   any size data can be processed!!
    data = np.memmap(fname, dtype='uint16', mode='r')
    data = data.reshape(-1, 512, 512)

    #
    corr_maxs = np.zeros(data.shape[0])
    shifts = np.zeros((data.shape[0],2))

    #
    subtract_flag = True

    #
    a = template.copy()
    if idx_parmap[0]==0:
        for idx in tqdm(idx_parmap, desc="phase corr computation"):

            # seelct an image
            #print ('idx: ', idx, data.shape)
            b = data[idx]

            #
            G_a = np.fft.fft2(a)
            G_b = np.fft.fft2(b)
            conj_b = np.ma.conjugate(G_b)
            R = G_a * conj_b
            R /= np.absolute(R)
            surface = np.fft.ifft2(R).real

            # compute peak location for row and column
            r, c = np.unravel_index(surface.argmax(), surface.shape)

            #
            corr_maxs[idx] = surface[r, c]

            # convert to roll function which has negative and positive values
            if subtract_flag:
                if r > 512 / 2:
                    r = r - 512

                if c > 512 / 2:
                    c = c - 512

            #
            shifts[idx] = [r, c]

    else:
        for idx in idx_parmap:

            # seelct an image
            # print ('idx: ', idx, data.shape)
            b = data[idx]

            #
            G_a = np.fft.fft2(a)
            G_b = np.fft.fft2(b)
            conj_b = np.ma.conjugate(G_b)
            R = G_a * conj_b
            R /= np.absolute(R)
            surface = np.fft.ifft2(R).real

            # compute peak location for row and column
            r, c = np.unravel_index(surface.argmax(), surface.shape)

            #
            corr_maxs[idx] = surface[r, c]

            # convert to roll function which has negative and positive values
            if subtract_flag:
                if r > 512 / 2:
                    r = r - 512

                if c > 512 / 2:
                    c = c - 512

            #
            shifts[idx] = [r, c]


    #
    return np.int32(shifts), corr_maxs



def make_template2(data,
                  fname_mmap,
                  n_imgs_to_sample = 500,
                  n_best_imgs = 100,
                  template = None,
                  idx_imgs = None,
                  random_img_sample_flag = True,
                  plotting=False,
                  n_cores=10):

    # find best correlation map first
    # don't pick random frames, much harder to find matchin frames

    if idx_imgs is None:
        if random_img_sample_flag:
            idx_imgs = np.random.choice(np.arange(data.shape[0]),
                                    n_imgs_to_sample,
                                    replace=False)
        else:
            idx_start = np.random.choice(np.arange(data.shape[0]-n_imgs_to_sample))
            idx_imgs = np.arange(idx_start,
                                 idx_start+n_imgs_to_sample,
                                 1)

    #
    #print ("idx imgs; ", idx_imgs)

    # make temporary template to match to
    if template is None:
        template = np.mean(data[idx_imgs],axis=0)

    # parallelize
    if n_cores==1:
        corr_maxs = np.zeros(idx_imgs.shape[0])
        ctr=0
        for k in tqdm(idx_imgs, desc="computing phase correlations"):

            #
            temp = phase_correlation(data[k], template)

            r,c = np.unravel_index(temp.argmax(), temp.shape)

            maxcorr = temp[r,c]
            corr_maxs[ctr] = maxcorr
            ctr+=1
    else:
        # split the image indexes into gropus
        imgs_split = np.array_split(idx_imgs,
                                    40)

        #
        res = parmap.map(phase_correlation_parallel2,
                         imgs_split,   # indexes of each image to process
                         template,     # defatul template
                         fname_mmap,   # place where to load data from
                         pm_pbar = True,
                         pm_processes = n_cores
                         )

        # initialize arrays
        shifts = np.zeros((data.shape[0], 2))
        corr_maxs = np.zeros(data.shape[0])

        # merge the shifts:
        for k in range(len(res)):
            shifts = shifts + res[k][0]
            corr_maxs = corr_maxs + res[k][1]

        # select only the values chose
        #shifts = shifts[idx_imgs]
        corr_maxs = corr_maxs[idx_imgs]

    #
    idx = np.argsort(corr_maxs)[::-1]

    # take the n best images and recompute template
    idx_best = idx[:n_best_imgs]
    template = data[idx_imgs[idx_best]].mean(0)

    #
    return corr_maxs, template, idx_imgs





In [3]:
#######################################################################
########### LOAD PRE-BMI DATA (e.g. 10-15mins recording) ##############
#######################################################################
#fname = '/home/cat/data/donato/bscope_tests/8499/data/Image_001_001.raw'
#fname = '/home/cat/data/donato/bscope_tests/8499/image_1000frames.raw'  # laptop
fname = '/home/cat/data/donato/bscope_tests/8499/data/Image_001_001_original_do_not_overwrite.raw'   # workstation
#fname = r'D:\bmi\DON8498\22-06-08\calibration\Image_001_001.raw'

data = np.fromfile(fname, dtype='uint16').reshape(-1, 512,512)
print (data.shape)

# show test image
plt.figure()
plt.imshow(data[100],vmax=np.max(data[100])/5.)
plt.show()


(10000, 512, 512)


In [4]:
#################################################################################################
############ FOLLOW SUITE2P: BUILD TEMPLATE FROM HIGHEST CORRELATED FRAMES FIRST ################
#################################################################################################


##################################################
import parmap 
n_imgs_to_sample = 5000
n_best_imgs = 500
corr_maxs, template, idx_imgs = make_template2(data, 
                                              fname,
                                              n_imgs_to_sample,
                                              n_best_imgs)

# 
plt.figure()
ax=plt.subplot(1,2,1)
temp = data[idx_imgs].mean(0)
plt.title("Average map over "+str(idx_imgs.shape[0])+" images")
plt.imshow(temp,
          #vmin=0,
          #vmax=1500,
          )


#
ax=plt.subplot(1,2,2)
plt.title("Average map over highest correlated "+str(n_best_imgs)+" images")
plt.imshow(template,
          #vmin=0,
          #vmax=1500,
          )

#
plt.show()

/home/cat/anaconda3/envs/bmi2/lib/python3.8/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 40/40 [00:29<00:00,  1.37it/s]


In [47]:
################################################################
############### TEST DRIFT ON ARTIFICIALL DATA #################
################################################################

# maximum shift in drift in pixels in each of the directoins (so 20 pixels = -20 .. +20)
x_max = 100
y_max = 100

#
x_shifts, y_shifts = get_drift_xy(data, x_max, y_max)
print (x_shifts, y_shifts)

# detect drift in template
drifts = []
maxcorrs = []

#for k in trange(data.shape[0]):
for k in trange(100):
    
    #
    img = np.zeros(data[0].shape)
    
    # take sample of data
    temp = template
    
    # add some noise
    noise = np.random.rand(temp.shape[0], temp.shape[1])*10000
   # print ("noise max min, ", np.min(noise), np.max(noise))
    #print ("template max min, ", np.min(temp), np.max(temp))
    temp = temp+noise
    
    
    # shift x
    temp = np.roll(temp, x_shifts[k], axis=0)
    temp = np.roll(temp, y_shifts[k], axis=1)
    
    #
    img_corr = phase_correlation(temp, template)

    r,c = np.unravel_index(img_corr.argmax(), img_corr.shape)

    maxcorr = img_corr[r,c]
    
    #
    maxcorrs.append(maxcorr)
    
    #
    drifts.append([r,c])
    
    
drifts = np.vstack(drifts)
idx = np.where(drifts>(512//2))
drifts[idx] = drifts[idx]-512

#
plt.figure()
ax=plt.subplot(1,2,1)
plt.plot(x_shifts[:100],label='true x shift')
plt.plot(drifts[:,0]+.1, label = 'detected x shift')
plt.legend()
plt.ylabel("pixel drift x", fontsize=20)
plt.xlabel("frame #", fontsize=20)

#
ax=plt.subplot(1,2,2)
plt.plot(y_shifts[:100],label='true y shift')
plt.plot(drifts[:,1]+.1, label = 'detected y shift')
plt.ylabel("pixel drift y", fontsize=20)
plt.xlabel("frame #", fontsize=20)

plt.legend()
plt.suptitle("Simulated drift detection (i.e. shifted template) - with 5 x gaussian noise")
plt.show()
print ("DONE")

[ 6 13 85 ... 73 90 50] [-82  69  14 ... -98  -8 -24]


100%|███████████████████████████████████████████████████████████████████| 100/100 [00:05<00:00, 18.55it/s]


DONE


In [5]:
##################################################
######## RUN MOTION CORRECTION ON REAL DATA ######
##################################################

#fname = '/home/cat/data_temp/cali2/Image_001_001.raw'   # workstation
#fname = r'D:\bmi\DON8498\22-06-08\calibration\Image_001_001.raw'

data = np.fromfile(fname, dtype='uint16').reshape(-1, 512,512)
print (data.shape)


# detect drift in data
drifts = []
maxcorrs = []

#
if False:
    n_imgs = 5000
    idx_imgs = np.random.choice(np.arange(data.shape[0]), n_imgs, replace=False) 
    template = np.median(data[idx_imgs], axis=0)
#mean(0)


# 
#for k in trange(data.shape[0]):
buffer = 5
c_last = 0
r_last = 0
max_jump = 10
for k in trange(data.shape[0]):
    
    #
    img = np.zeros(data[0].shape)
    
    # take sample of data
    temp = data[k:k+buffer]
    temp = np.mean(temp,axis=0)
    
    #
    img_corr = phase_correlation(temp, template)

    r,c = np.unravel_index(img_corr.argmax(), img_corr.shape)

    #
    if r > 512 / 2:
        r = r - 512

    if c > 512 / 2:
        c = c - 512

    #
    if abs(r)>max_jump:
        r=r_last
    if abs(c)>max_jump:
        c=c_last
    
    #
    maxcorr = img_corr[r,c]
    
    #
    maxcorrs.append(maxcorr)
    
    #
    drifts.append([r,c])
    
    #
    r_last = r
    c_last = c
    
#####################################################################
#####################################################################
#####################################################################

# convert to array
drifts = np.vstack(drifts)
idx = np.where(drifts>(512//2))
drifts[idx] = drifts[idx]-512


(10000, 512, 512)


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 10000/10000 [03:11<00:00, 52.11it/s]


DONE


In [10]:

#
t = np.arange(drifts.shape[0])/30.
plt.figure()
ax=plt.subplot(2,1,1)
plt.plot(t,drifts[:,0]+.1, label = 'detected x shift')
plt.ylabel("pixels", fontsize=20)
plt.legend()

ax=plt.subplot(2,1,2)
plt.plot(drifts[:,1]+.1, label = 'detected y shift')
plt.ylabel("pixels", fontsize=20)
plt.xlabel("Time(sec)", fontsize=20)
plt.legend()
plt.suptitle("Drift detection" + fname)
plt.show()
print ("DONE")


DONE


In [101]:
########################################################################
############### MAKE PRE and POST DRIFT CORRECTION MOVIES ##############
########################################################################
import cv2
from scipy.ndimage import gaussian_filter

# 
def make_video(fname_out,
               data,
               smooth_flag=False,
               low_noise_flag=False,
               fps=30):


    # load and videos
    size = [512,512]
    fps = 25
    
    # 
    out = cv2.VideoWriter(fname_out, 
                          cv2.VideoWriter_fourcc(*'mp4v'), 
                          fps, (size[1], size[0]), 
                          True)
    
    #
    for k in trange(data.shape[0]):
        # 
        temp = data[k]
        
        #
        if smooth_flag:
            temp = gaussian_filter(temp, sigma=1.5)

        # can also use np.tile()
        frame = [temp,temp,temp]
        frame = np.array(frame).T
        
        # 
        frame = cv2.normalize(frame, None, 0, 255, cv2.NORM_MINMAX, cv2.CV_8U)
                
        #
        if low_noise_flag:
            idx = np.where(temp<128)
            temp[idx]=0
                   
        out.write(frame)
    out.release()
        
        
#
def shift_frame(data, x, y):
    
    data = np.roll(data, x, axis=0)
    data = np.roll(data, y, axis=1)
    
    return data
#
corrected_movie = np.zeros(data.shape)
for k in trange(drifts.shape[0]):
    
    x = -drifts[k,0]
    y = -drifts[k,1]
    
    frame = data[k]
    
    frame_corrected = shift_frame(frame, x, y)
    
    corrected_movie[k] = frame_corrected

#
smooth_flag = True
low_noise_flag = False
#fname_out='/home/cat/original_vid.mp4'
fname_out='/home/cat/corrected_vid.mp4'
make_video(fname_out,
           #data[500:1000],
           corrected_movie[500:1000],
           smooth_flag,
           low_noise_flag
          )

print ("done..")




100%|███████████████████████████████████████████████████████████████████| 500/500 [00:12<00:00, 40.46it/s]


done..
